# Generic optimisation run notebook

Canonical notebook contract for users and future AI assistants: copy this structure unchanged for every experiment. Only edit the brief experiment description, `run_name`, and argument values. Keep executable cells argument-only; do not add notebook-local functions, bespoke imports, subprocess logic, query logic, plotting helpers, or case-specific essays. Reusable behavior belongs in `ofc.notebook_workflow` or the relevant package module.

The config, local-run, and Slurm sections are optional and independent; skip them entirely when data already exists. They are inert until their first-line `Activated` flag is deliberately changed to `True`. After data exists, a fresh kernel needs only setup, query, and the desired plot cells. The query can inherit the existing YAML automatically or ignore local configs and select older runs directly from the results database.

Experiment: N=100, `u_max=160` sweep over smoothness and sharpness using the first-listed Adam settings and a 10,000-step schedule.

In [ ]:
from ofc.notebook_workflow import RunNotebook

run_name = "N100_u160_smoothness_sharpness_sweep_10k"
workflow = RunNotebook(run_name)
workflow.show_context()

## Create the immutable config

Edit the arguments, then activate once. Returning later: leave `Activated=False`; the existing YAML is loaded without being changed.

In [ ]:
Activated = False

description = "N=100 u_max=160 smoothness and sharpness sweep with fixed first-listed Adam settings."
reuse_existing = False
parameters = {
    "N": 100,
    "t_interval": 4.0,
    "r_bg": -0.008716,
    "u_isbound": True,
    "v_isbound": True,
    "u_max": 160.0,
    "v_max": 1000.0,
    "slew_limit": 0.05,
    "optimizer": "adam",
    "schedule": [(1_000, 1.0), (2_000, 0.1), (7_000, 0.1)],
    "adam_learning_rate": 0.05,
    "adam_beta1": 0.9,
    "adam_beta2": 0.999,
    "adam_eps": 1e-8,
    "lbfgs_history_size": 100,
    "lbfgs_max_linesearch_steps": 200,
    "lbfgs_tolerance": 1e-6,
    "smoothness": [2e-6, 1e-6, 0.5e-6, 0.25e-6, 0.125e-6, 0.0625e-6],
    "u_smooth": None,
    "v_smooth": None,
    "sharpness": [10e-8, 5e-8, 2.5e-8, 1.25e-8, 0.625e-8],
    "u_sharp": None,
    "v_sharp": None,
    "block_size": 500,
    "J_tol": 1e-5,
    "u_tol": 1e-3,
    "v_tol": 1e-3,
}
runtime = {
    "initialisations": 10,
    "fourier_num_modes": 5,
    "fourier_rms_amplitude": 0.3,
    "fourier_intensity_fraction": 0.3,
    "use_jit": True,
    "use_x64": True,
    "device": "cpu",
    "concurrent_workers": 1,
    "max_cases_per_batch": 5,
    "database": "results/results.sqlite3",
}
initialization_query = {
    "where": {
        "config_id": 7388691636552290904,
        "queue_id": 696482,
        "status": "complete",
        "N": 100,
        "u_max": 160.0,
    },
    "limit": 1,
    "order_by": "best_score",
    "descending": True,
    "control_kind": "best",
    "perturbed": True,
    "perturbation_levels": [0.0025, 0.005, 0.01, 0.025, 0.05],
    "match_parameters": ["u_max"],
}

config_document = workflow.create_config(
    activated=Activated,
    description=description,
    parameters=parameters,
    runtime=runtime,
    initialization_query=initialization_query,
    reuse_existing=reuse_existing,
)

## Run directly on `bar`'s GPU (detached)

This launches outside Slurm, verifies that JAX can see the GPU, and detaches the process into its own session and log so it survives a browser or laptop disconnect. The immutable config must use `device: auto` or `device: gpu`.

In [ ]:
Activated = False

queue_id = None  # None creates a random ID; otherwise use a positive integer.
python_executable = None  # None uses this kernel's Python; otherwise give a path.
extra_arguments = []  # Example: ["--batch-index", "0"].
detached = True  # Keep running if the notebook/browser disconnects.
log_path = None  # None writes logs/<run-name>-local-<queue-id>.log.

active_queue_id = workflow.run_on_bar_gpu(
    activated=Activated,
    queue_id=queue_id,
    python_executable=python_executable,
    extra_arguments=extra_arguments,
    detached=detached,
    log_path=log_path,
)

## Submit through Slurm (alternative)

In [ ]:
Activated = False

partition = "zen5,epyc"
time = "00:10:00"
cpus = 2
memory = "1G"
array = True
array_max_concurrent = None
job_name = None
extra_arguments = []

active_queue_id = workflow.submit_slurm(
    activated=Activated,
    partition=partition,
    time=time,
    cpus=cpus,
    memory=memory,
    array=array,
    array_max_concurrent=array_max_concurrent,
    job_name=job_name,
    extra_arguments=extra_arguments,
)

## Query persisted data

This is read-only and does not depend on executing any optional cell above. It supports any number of varying parameters; choose the dimensions used by each plot later. With config inheritance enabled, the existing YAML automatically supplies its database and immutable config identity (and the selected rows carry all resolved config parameters). Disable inheritance to query older runs using only database filters.

In [ ]:
inherit_config = True  # True = use run_config/<run_name>.yaml automatically; False = historical database only.
database = None  # None = inherited config database, or results/results.sqlite3 when inheritance is False.
queue_id = None  # None = latest matching execution; or an integer such as 700843.
config_run_rank = 1  # 1 = latest, 2 = second latest, etc.
statuses = None  # None = all; or "running", "complete", "failed", or a list.
# With inherit_config=False, identify old data here, e.g. {"config_name": "old_run"}.
filters = {}  # Exact: {"N": 100}; list: {"u_max": [40, 160]}; range: {"best_score": (0, 100)}.
# Any number of dimensions. None discovers all varying parameters automatically.
# Names: N, t_interval, r_bg, u_isbound, v_isbound, u_max, v_max, slew_limit,
# optimizer, schedule, adam_learning_rate, adam_beta1, adam_beta2, adam_eps,
# lbfgs_history_size, lbfgs_max_linesearch_steps, lbfgs_tolerance, smoothness,
# u_smooth, v_smooth, sharpness, u_sharp, v_sharp, block_size, J_tol, u_tol, v_tol.
sweep_parameters = ["smoothness", "sharpness"]
require_saved_stage = True  # True excludes registered runs with no saved stage yet.
limit = None  # None = unlimited; otherwise a positive integer.
order_by = "run_id"  # Examples: "run_id", "best_score", "u_max", "started_utc".
descending = False

query_result = workflow.query(
    inherit_config=inherit_config,
    database=database,
    queue_id=queue_id,
    config_run_rank=config_run_rank,
    statuses=statuses,
    filters=filters,
    sweep_parameters=sweep_parameters,
    require_saved_stage=require_saved_stage,
    limit=limit,
    order_by=order_by,
    descending=descending,
)

## Figure display and saving

In [ ]:
save_figure = None  # None = display only; "/" = figures/; or "/experiment/latest".
figure_format = "png"  # "png" or "pdf".
preview_dpi = 180  # Increase this if inline multi-sweep labels are hard to read.
save_dpi = 600  # Resolution used when saving PNG figures.

## Figure 1 — convergence

In [ ]:
sweep_parameter = "smoothness"
figure_1 = query_result.plot_convergence(
    sweep_parameter=sweep_parameter,
    log_base_x=10,
    log_base_y=2,
    base_x=None,
    base_y=None,
    x_multiplier=1,
    y_multiplier=1,
    x_range=None,
    y_range=None,
    x_label=None,
    y_label=None,
)
workflow.present_figure(
    figure_1, "01_convergence", save_figure=save_figure, figure_format=figure_format
)

## Figure 2 — objective strip plot (equally spaced sweep values) and seed sensitivity

In [ ]:
sweep_parameter = "smoothness"
figure_2 = query_result.plot_distribution(
    sweep_parameter=sweep_parameter,
    log_base_y=10,
    base_y=None,
    y_multiplier=1,
    y_range=None,
    x_label=None,
    y_label=None,
    point_size=12,
    line_alpha=0.22,
    seed_sensitivity_log_base_y=10,
    seed_sensitivity_base_y=None,
    seed_sensitivity_y_multiplier=1,
    seed_sensitivity_y_range=None,
    seed_sensitivity_tolerance=None,  # None uses stored J_tol; or use 0.1 / [0.05, 0.1].
)
workflow.present_figure(
    figure_2, "02_distribution", save_figure=save_figure, figure_format=figure_format
)

## Figure 3 — best controls

In [ ]:
sweep_parameter = "smoothness"
figure_3 = query_result.plot_controls(
    sweep_parameter=sweep_parameter,
    log_base_x=None,
    log_base_y=None,
    base_x=None,
    base_y=None,
    x_multiplier=1,
    y_multiplier=1,
    x_range=None,
    y_range=None,
    x_label=None,
    y_label=None,
)
workflow.present_figure(
    figure_3, "03_controls", save_figure=save_figure, figure_format=figure_format
)

## Single sweep summary

In [ ]:
single_sweep_parameter = "smoothness"
history_points = 1200
single_sweep_figure = query_result.plot_single_sweep_summary(
    sweep_parameter=single_sweep_parameter,
    history_points=history_points,
)
workflow.present_figure(
    single_sweep_figure,
    "04_single_sweep_summary",
    save_figure=save_figure,
    figure_format=figure_format,
    preview_dpi=preview_dpi,
    save_dpi=save_dpi,
)

## Double sweep summary

In [ ]:
separate_sweep_parameter = "smoothness"
colour_sweep_parameter = "sharpness"
history_points = 1200
double_sweep_figure = query_result.plot_double_sweep_summary(
    separate_sweep_parameter=separate_sweep_parameter,
    colour_sweep_parameter=colour_sweep_parameter,
    history_points=history_points,
)
workflow.present_figure(
    double_sweep_figure,
    "05_double_sweep_summary",
    save_figure=save_figure,
    figure_format=figure_format,
    preview_dpi=preview_dpi,
    save_dpi=save_dpi,
)

## Triple sweep summary

In [ ]:
row_sweep_parameter = "u_max"
column_sweep_parameter = "smoothness"
colour_sweep_parameter = "sharpness"
history_points = 1200
triple_sweep_figure = query_result.plot_triple_sweep_summary(
    row_sweep_parameter=row_sweep_parameter,
    column_sweep_parameter=column_sweep_parameter,
    colour_sweep_parameter=colour_sweep_parameter,
    history_points=history_points,
)
workflow.present_figure(
    triple_sweep_figure,
    "06_triple_sweep_summary",
    save_figure=save_figure,
    figure_format=figure_format,
    preview_dpi=preview_dpi,
    save_dpi=save_dpi,
)